# Ecommerce Customer Segmentation And Prediction

## Objectives:
1. Enhance marketing strategies and customer retention by identifying distinct customer segments.
2. Use machine learning to segment customers and predict future purchase activity


## Read and Preprocess Data

In [4]:
import pandas as pd
import numpy as np

In [41]:
sales_data = pd.read_csv('data/data.csv', encoding='latin-1')
sales_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [42]:
sales_data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


## Data Preprocessing

### Check Null Values

In [43]:
sales_data.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [44]:
customerID_missing_percentage = sales_data.CustomerID.isna().sum() / len(sales_data) * 100
print(f"Missing data percentage (for customerID): {customerID_missing_percentage:.2f}%")

Missing data percentage (for customerID): 24.93%


In [88]:
sales_data.CustomerID.value_counts()

CustomerID
17841.0    7983
14911.0    5903
14096.0    5128
12748.0    4642
14606.0    2782
           ... 
18184.0       1
13256.0       1
13017.0       1
18174.0       1
15195.0       1
Name: count, Length: 4372, dtype: int64

Records which have no Customer ID and no Description are likely to be zero-spend null records.

In [ ]:
def describe_description_customer_nulls(df, target_cols):
    """Count nulls and flag rows missing both Description and CustomerID."""
    missing_counts = df[target_cols].isna().sum()
    for col, count in missing_counts.items():
        print(f"{col} null records: {count}")

    null_records_mask = df[target_cols].isna().all(axis=1)
    common_zerovalue_records = df[null_records_mask]
    print(f"Records with both Description and CustomerID null: {len(common_zerovalue_records)}")

    
    cleaned_df = df[~null_records_mask].copy()
    print("Deleted zero-spend null records")
    return common_zerovalue_records, cleaned_df


In [ ]:
_  , cleaned_df = describe_description_customer_nulls(sales_data, ["Description", "CustomerID"])

Description null records: 1454
CustomerID null records: 135080
Records with both Description and CustomerID null: 1454
Deleted zero-spend null records


In [71]:
cleaned_df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


In [72]:
cleaned_df.isna().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     133626
Country             0
dtype: int64

Map CustomerID with InvoiceNo to identify how many customers have **multiple transactions** and better understand customer *spending patterns*.

In [73]:
def classify_customer_orders(df: pd.DataFrame):
    """Classify customers based on number of unique invoices."""
    
    customer_invoice_counts = (
        df.groupby("CustomerID")["InvoiceNo"]
          .nunique()
          .reset_index(name="invoice_count")
    )
    
    customer_invoice_counts["order_type"] = customer_invoice_counts["invoice_count"].apply(
        lambda x: "single_invoice" if x == 1 else "multiple_invoices"
    )
    
    return customer_invoice_counts

In [74]:
customer_invoice_map_df = classify_customer_orders(cleaned_df)
customer_invoice_map_df.order_type.value_counts()

order_type
multiple_invoices    3059
single_invoice       1313
Name: count, dtype: int64

This means a customer has multiple invoices associated with them than a one-to-one relationship.

In [75]:
customer_missing_mask = cleaned_df.CustomerID.isna()
customer_missing_df = cleaned_df[customer_missing_mask].copy()
customer_missing_df.to_csv('data/unknown_customer_data.csv',sep=',')

In [79]:
cleaned_df['CustomerID'] = cleaned_df.CustomerID.fillna(0).astype('Int64')

Fill na values in the 'CustomerID' column with 0 and convert the column to integer type for easier analysis.

In [80]:
cleaned_df.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [86]:
print("Number of unique customers :", len(list(cleaned_df.CustomerID.value_counts())))

Number of unique customers : 4373


In [87]:
print("Number of total countries :", len(list(cleaned_df.Country.value_counts())))

Number of total countries : 38


In [ ]:
print("Number of missing unit prices :", cleaned_df.UnitPrice.eq(0.0).sum())

Number of missing unit prices : 0


### Feature Engineering

In [92]:
cleaned_df["InvoiceDate"] = pd.to_datetime(cleaned_df["InvoiceDate"])
cleaned_df["TotalAmount"] = cleaned_df["Quantity"] * cleaned_df["UnitPrice"]

In [93]:
cleaned_df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60


### Exploratory Data Analysis (EDA)

In [ ]:
#TODO EDA, Feature Engineering (RFM, Avg Value), Model Building,